# AIDEN Multimodal Service — Colab Proxy

Run this notebook to host LLaVA / Qwen-VL inference behind an ngrok tunnel.
AIDEN's backend will use `MULTIMODAL_REMOTE_URL` to forward vision requests here.

**⏱ Setup time:** ~5 min (mostly model download)
**💾 Disk:** ~8 GB for LLaVA or ~5 GB for Qwen-VL
**🧠 RAM:** Colab T4 High-RAM recommended

## Step 1 — Install Dependencies

In [ ]:
# ── Install system + Python deps ──────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq ngrok > /dev/null 2>&1
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers>=4.56.2 accelerate>=1.4.0 sentence-transformers fastapi uvicorn pillow pyngrok

print("Dependencies installed.")

## Step 2 — Mount Google Drive & Clone Repo

Run this cell to mount your Drive and clone your repository.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a project directory in Drive
PROJECT_DIR = "/content/drive/MyDrive/aiden_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# 3. Navigate to the project directory
%cd {PROJECT_DIR}

# 4. Clone the repository (first time only)
!git clone https://github.com/Bharath8220818/aiden.git

# 5. Navigate into the backend folder
%cd aiden/backend

## Step 3 — Start ngrok Tunnel

Get your authtoken from https://dashboard.ngrok.com/auth

In [ ]:
import getpass

NGROK_AUTH_TOKEN = getpass.getpass("Enter your ngrok authtoken: ")

# Configure ngrok
!ngrok authtoken $NGROK_AUTH_TOKEN

# Start tunnel on port 8001
import subprocess
import threading
import time

def start_ngrok():
    subprocess.run(["ngrok", "http", "8001", "--log", "stdout"])

threading.Thread(target=start_ngrok, daemon=True).start()
time.sleep(3)
print("ngrok tunnel started.")

In [ ]:
import requests

# Get the public ngrok URL
try:
    r = requests.get("http://localhost:4040/api/tunnels")
    public_url = r.json()["tunnels"][0]["public_url"]
    print(f"\n=== NGROK URL: {public_url} ===\n")
    print("Set this as MULTIMODAL_REMOTE_URL in your .env or Render dashboard.")
except Exception as e:
    print(f"Could not fetch ngrok URL: {e}")

## Step 4 — Start the Multimodal Inference Server

This loads the LLaVA model (7B) and serves the `/analyze` endpoint.

In [ ]:
import subprocess
import sys

# ── Start the multimodal proxy server ─────────────────────────────────
# The server listens on port 8001 and exposes:
#   POST /analyze  — accepts {"image": "<base64>", "prompt": "..."}
#   GET  /health   — returns 200 when ready

cmd = [
    sys.executable, "-m", "uvicorn",
    "app.services.colab_proxy",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--log-level", "info",
]

print("Loading model (this takes ~2-3 minutes)...")
proc = subprocess.Popen(cmd)
print(f"Server PID: {proc.pid}")
print("\nServer is running! Keep this cell running.")

## Test the Endpoint (Optional)

Run this in a separate cell to verify the server is working.

In [ ]:
import requests

# Health check
r = requests.get("http://localhost:8001/health")
print(f"Health: {r.status_code} - {r.json()}")